## Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
from tqdm import tqdm
import tensorflow as tf
from pathlib import Path

from csv import QUOTE_NONE

from recommenders.models.newsrec.newsrec_utils import prepare_hparams, get_mind_data_set
from recommenders.models.newsrec.models.nrms import NRMSModel
from recommenders.models.newsrec.io.mind_iterator import MINDIterator

print("System version: {}".format(sys.version))
print("Tensorflow version: {}".format(tf.__version__))

System version: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
System version: 3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]
Tensorflow version: 2.20.0


In [38]:
epochs = 5
seed = 42
batch_size = 32

# Options: demo, small, large
MIND_type = 'small'

# Root folder containing the extracted MIND splits
LOCAL_DATA_ROOT = Path(os.getenv('MIND_DATA_ROOT', 'smallDataset')).expanduser()


## Load

In [ ]:

LOCAL_DATA_ROOT = Path(os.environ.get("MIND_DATA_ROOT", "./smallDataset")).expanduser().resolve()

train_dir = LOCAL_DATA_ROOT / f"MINDsmall_train"
valid_dir = LOCAL_DATA_ROOT / f"MIND_dev"

train_news_path      = str(train_dir / "news.tsv")
train_behaviors_path = str(train_dir / "behaviors.tsv")
valid_news_path      = str(valid_dir / "news.tsv")
valid_behaviors_path = str(valid_dir / "behaviors.tsv")

# 3) Load TSVs into DataFrames used throughout the notebook
NEWS_COLUMNS = [
    "nid",
    "vertical",
    "subvertical",
    "title",
    "abstract",
    "url",
    "title_entities",
    "abstract_entities",
]
BEHAVIOR_COLUMNS = [
    "impression_id",
    "user_id",
    "time",
    "history",
    "impressions",
]

def load_tsv(path_str: str, columns):
    return pd.read_table(
        path_str,
        header=None,
        names=columns,
        sep='\t',
        quoting=QUOTE_NONE,
        dtype=object,
        na_filter=False,
    )

train_news_df = load_tsv(train_news_path, NEWS_COLUMNS)
valid_news_df = load_tsv(valid_news_path, NEWS_COLUMNS)
train_behaviors_df = load_tsv(train_behaviors_path, BEHAVIOR_COLUMNS)
valid_behaviors_df = load_tsv(valid_behaviors_path, BEHAVIOR_COLUMNS)

print(f"Loaded {len(train_news_df)} train news rows and {len(valid_news_df)} validation news rows from {LOCAL_DATA_ROOT}")
print(f"Loaded {len(train_behaviors_df)} train behaviors rows and {len(valid_behaviors_df)} validation behaviors rows")


Loaded 51282 train news rows and 42416 validation news rows from /home/asmvas/Documents/NTNU-prosjekt/anbefalingssytem/AnbSysProsjekt/smallDataset
Loaded 156965 train behaviors rows and 73152 validation behaviors rows


## Hyperparams

In [ ]:
hparams = prepare_hparams(yaml_file, 
                          wordEmb_file=wordEmb_file,
                          wordDict_file=wordDict_file, 
                          userDict_file=userDict_file,
                          batch_size=batch_size,
                          epochs=epochs,
                          show_step=10)
print(hparams)

## Train model